# INSTALL & IMPORT LIBRARY

In [ ]:
!pip install newspaper3k
!pip install Sastrawi
!pip install Stopword
!pip install nltk
!pip install wordcloud
!pip install lxml_html_clean

# IMPORT DATA

In [ ]:
import newspaper
import pandas as pd
import nltk

from newspaper import news_pool
from newspaper import Article


nltk.download('punkt_tab')
nltk.download('punkt')

# Reading the Excel file
df = pd.read_csv("berita lokal - LABEL DETIK.csv")

In [ ]:
df.head()

In [ ]:
import re

# Daftar kata yang ingin dihapus
custom_stop_words = {'advertisement', 'scroll', 'continue', 'with', 'content', 'to'}

# Fungsi untuk menghapus kata-kata tertentu
def hapus_custom_stop_words(teks):
    # Ubah teks menjadi lowercase dan hilangkan kata dalam custom_stop_words
    teks = ' '.join(word for word in str(teks).lower().split() if word not in custom_stop_words)
    return teks

# Terapkan fungsi ke kolom 'Teks'
df['Teks_bersih'] = df['Teks'].apply(hapus_custom_stop_words)

# Tambahkan kolom untuk menghitung total kata di setiap berita
df['total_kata'] = df['Teks_bersih'].apply(lambda x: len(str(x).split()))

# Total keseluruhan kata
total_kata = df['total_kata'].sum()

# Hitung rata-rata kata per berita
average_kata = total_kata / len(df)

# Output hasil
print(f"Total kata di semua berita: {total_kata}")
print(f"Rata-rata kata per berita: {average_kata:.2f}")
print(df[['Teks', 'Teks_bersih', 'total_kata']].head())

In [ ]:
# Konversi kolom 'Tanggal publikasi' ke format datetime jika belum
df['Tanggal publikasi'] = pd.to_datetime(df['Tanggal publikasi'])

# Filter data hanya untuk bulan Februari
df_okt = df[df['Tanggal publikasi'].dt.month == 10]

# Tampilkan hasil
print("Data yang dirilis di bulan Februari:")
print(df_okt)

# PREPO

In [ ]:
import re
from nltk.corpus import stopwords
import nltk

# Unduh stopwords jika belum
nltk.download("stopwords")

# Ambil daftar stopwords
stop_words = set(stopwords.words('indonesian'))

# Tambahkan kata-kata tambahan yang ingin dihapus
custom_stop_words = {'advertisement', 'scroll', 'continue', 'with', 'content', 'to', 'generative', 'ai', '2024', 'penggunaan' ,'hasil' , 'model' , 'gen'}

# Gabungkan stopwords bawaan dengan custom stopwords
all_stop_words = stop_words.union(custom_stop_words)

# Fungsi untuk membersihkan teks
def bersihkan_teks(teks):
    teks = re.sub(r'\W+', ' ', teks.lower())  # Hilangkan simbol dan jadikan lowercase
    teks = ' '.join(word for word in teks.split() if word not in all_stop_words)  # Hilangkan stopwords
    return teks

# Terapkan fungsi ke teks
df['Teks_Prepo'] = df['Teks'].apply(bersihkan_teks)

# Update analisis word cloud atau top kata dengan teks bersih
all_text_bersih = ' '.join(df['Teks_Prepo'])

# EDA

In [ ]:
# Ubah semua label menjadi lowercase
df['Label'] = df['Label'].str.lower()

# Jumlah berita per label
jumlah_berita_per_label = df['Label'].value_counts()
print(jumlah_berita_per_label)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Gabungkan semua teks menjadi satu string
all_text = ' '.join(df['Teks_Prepo'])

# Buat Word Cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)

# Tampilkan Word Cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud Keseluruhan")
plt.show()

In [ ]:
from collections import Counter
import pandas as pd

# Pisahkan semua kata dari teks
words = all_text.split()

# Hitung frekuensi kata
word_freq = Counter(words)

# Konversi ke DataFrame
word_freq_df = pd.DataFrame(word_freq.items(), columns=['Kata', 'Frekuensi']).sort_values(by='Frekuensi', ascending=False)

# Top 10 kata
top_10_kata = word_freq_df.head(10)
print(top_10_kata)


In [ ]:
# Konversi kolom ke format datetime
df['Tanggal publikasi'] = pd.to_datetime(df['Tanggal publikasi'])

# Filter hanya tahun 2024
df_2024 = df[df['Tanggal publikasi'].dt.year == 2024]

# Tambahkan kolom bulan
df_2024['Bulan'] = df_2024['Tanggal publikasi'].dt.month

# Hitung distribusi per bulan
distribusi_per_bulan = df_2024['Bulan'].value_counts().sort_index()

# Konversi hasil ke DataFrame untuk visualisasi
distribusi_df = distribusi_per_bulan.reset_index()
distribusi_df.columns = ['Bulan', 'Jumlah Berita']

print(distribusi_df)

In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from collections import Counter

# Daftar kata tambahan untuk dihapus (custom stopwords)
custom_stop_words = {'link', 'https', 'www', 'ai', 'advertisement', 'scroll', 'to', 'with', 'content', 'gen'}

# Daftar frasa yang ingin dihapus
phrases_to_remove = {'advertisement scroll', 'ai generatif', 'to continue', 'continue with', 'scroll to', 'with content'}

# Gabungkan semua stopwords
all_stop_words = stop_words.union(custom_stop_words)

# Fungsi untuk membersihkan teks
def bersihkan_teks(teks):
    teks = re.sub(r'https?://\S+|www\.\S+', '', teks)  # Hilangkan tautan
    teks = re.sub(r'\d+', '', teks)  # Hilangkan angka
    teks = re.sub(r'\W+', ' ', teks.lower())  # Hilangkan simbol dan jadikan lowercase
    teks = ' '.join(word for word in teks.split() if word not in all_stop_words)  # Hilangkan stopwords
    return teks

# Terapkan fungsi pembersihan
df['Teks_bersih'] = df['Teks'].apply(bersihkan_teks)

# Fungsi untuk menghitung n-gram
def hitung_ngram(teks, n=2):
    tokens = word_tokenize(teks)
    ngram_list = list(ngrams(tokens, n))
    # Gabungkan n-gram menjadi string agar mudah dibandingkan
    ngram_list = [' '.join(gram) for gram in ngram_list]
    # Hapus n-gram yang ada dalam daftar phrases_to_remove
    ngram_list = [ngram for ngram in ngram_list if ngram not in phrases_to_remove]
    return ngram_list

# Hitung top 10 2-gram per label dengan persentase
top_2gram_per_label = {}
for label, grup in df.groupby('Label'):
    all_teks = ' '.join(grup['Teks_bersih'])  # Gabungkan semua teks dalam satu label
    ngram_list = hitung_ngram(all_teks, n=2)  # Buat 2-gram
    frekuensi = Counter(ngram_list)  # Hitung frekuensi
    total_ngrams = sum(frekuensi.values())  # Hitung total semua 2-gram
    top_2gram = [(gram, freq, (freq / total_ngrams) * 100) for gram, freq in frekuensi.most_common(10)]  # Tambahkan persentase
    top_2gram_per_label[label] = top_2gram

# Tampilkan hasil
for label, top_2gram in top_2gram_per_label.items():
    print(f"Top 10 2-gram untuk label '{label}':")
    for gram, freq, percentage in top_2gram:
        print(f"- {gram}: {freq} ({percentage:.2f}%)")
    print()


In [ ]:
# Fungsi untuk menghitung n-gram
def hitung_ngram(teks, n=3):  # Ganti n=3 untuk 3-gram
    tokens = word_tokenize(teks)
    ngram_list = list(ngrams(tokens, n))
    # Gabungkan n-gram menjadi string agar mudah dibandingkan
    ngram_list = [' '.join(gram) for gram in ngram_list]
    # Hapus n-gram yang ada dalam daftar phrases_to_remove
    ngram_list = [ngram for ngram in ngram_list if ngram not in phrases_to_remove]
    return ngram_list

# Hitung top 10 3-gram per label
top_3gram_per_label = {}
for label, grup in df.groupby('Label'):
    all_teks = ' '.join(grup['Teks_bersih'])  # Gabungkan semua teks dalam satu label
    ngram_list = hitung_ngram(all_teks, n=3)  # Buat 3-gram
    frekuensi = Counter(ngram_list)  # Hitung frekuensi
    top_3gram = frekuensi.most_common(10)  # Ambil 10 3-gram teratas
    top_3gram_per_label[label] = top_3gram

# Tampilkan hasil
for label, top_3gram in top_3gram_per_label.items():
    print(f"Top 10 3-gram untuk label '{label}':")
    for gram, freq in top_3gram:
        print(f"- {gram}: {freq}")
    print()